# Model 7: XGBoost Multi-Quantile Regressor for Drug `N05B`

## Hyperparameter Selection Methodology:
Embedded Live 25-Trial Optuna Quantile Bayesian Optimization Study.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'N05B'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for N05B loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Live Optuna Quantile Hyperparameter Study Code
import optuna
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)

def create_enhanced_features(series):
    df_feat = pd.DataFrame(index=series.index)
    sales = series.values
    df_feat['sales'] = sales
    for lag in [1, 2, 3, 7, 14, 21, 28, 60, 90, 365]:
        df_feat[f'lag_{lag}'] = df_feat['sales'].shift(lag)
    for window in [7, 14, 28]:
        df_feat[f'rolling_mean_{window}'] = df_feat['sales'].shift(1).rolling(window).mean()
        df_feat[f'rolling_std_{window}']  = df_feat['sales'].shift(1).rolling(window).std()
        df_feat[f'rolling_max_{window}']  = df_feat['sales'].shift(1).rolling(window).max()
        df_feat[f'rolling_min_{window}']  = df_feat['sales'].shift(1).rolling(window).min()
    df_feat['ewm_mean_7']  = df_feat['sales'].shift(1).ewm(span=7).mean()
    df_feat['ewm_mean_28'] = df_feat['sales'].shift(1).ewm(span=28).mean()
    df_feat['cv_7'] = df_feat['rolling_std_7'] / (df_feat['rolling_mean_7'] + 1e-5)
    dof = df_feat.index.dayofyear
    dow = df_feat.index.dayofweek
    df_feat['sin_dayofyear'] = np.sin(2 * np.pi * dof / 365.25)
    df_feat['cos_dayofyear'] = np.cos(2 * np.pi * dof / 365.25)
    df_feat['sin_dayofweek'] = np.sin(2 * np.pi * dow / 7.0)
    df_feat['cos_dayofweek'] = np.cos(2 * np.pi * dow / 7.0)
    df_feat['dayofweek']     = dow
    df_feat['month']         = df_feat.index.month
    df_feat['dayofyear']     = dof
    df_feat['is_weekend']    = (dow >= 5).astype(float)
    df_feat['is_month_start'] = df_feat.index.is_month_start.astype(float)
    df_feat['is_month_end']   = df_feat.index.is_month_end.astype(float)
    return df_feat.drop(columns=['sales'])

feat_full = create_enhanced_features(full_series)

X_tr = feat_full.loc[train_series.index].dropna()
y_tr = np.log1p(train_series.loc[X_tr.index])
X_va = feat_full.loc[val_series.index].fillna(0)

X_cb_xgb = feat_full.loc[combined_series.index].dropna()
y_cb_xgb = np.log1p(combined_series.loc[X_cb_xgb.index])
y_cb_raw = combined_series.loc[X_cb_xgb.index]
X_ts_xgb = feat_full.loc[test_series.index].fillna(0)

def xgb_objective(trial):
    params = {
        'objective': 'reg:quantileerror',
        'quantile_alpha': 0.50,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr)
    pred_log = model.predict(X_va)
    pred = np.clip(np.expm1(pred_log), 0, None)
    return evaluate_metrics(val_series.values, pred)['RMSLE']

xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=25)

best_xgb_params = xgb_study.best_params
print("=== OPTUNA BAYESIAN OPTIMIZATION DISCOVERED PARAMETERS (XGBoost Quantile N05B) ===")
for k, v in best_xgb_params.items():
    print(f"  * {k:20s}: {v}")


=== OPTUNA BAYESIAN OPTIMIZATION DISCOVERED PARAMETERS (XGBoost Quantile N05B) ===
  * max_depth           : 3
  * learning_rate       : 0.014017307423237188
  * n_estimators        : 100
  * min_child_weight    : 9.108219462582262
  * subsample           : 0.9976016199916005
  * colsample_bytree    : 0.888388676781658
  * reg_alpha           : 1.1713300564723727e-08
  * reg_lambda          : 0.0003625232665571924


In [3]:
# Step 2: Fit Quantiles & Evaluate 2019 Test Holdout
m7_p50 = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.50, **best_xgb_params, random_state=42, n_jobs=-1)
m7_p50.fit(X_cb_xgb, y_cb_xgb)
pred_p50 = np.clip(np.expm1(m7_p50.predict(X_ts_xgb)), 0, None)

m7_p10 = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.10, **best_xgb_params, random_state=42, n_jobs=-1)
m7_p10.fit(X_cb_xgb, y_cb_xgb)
pred_p10 = np.clip(np.expm1(m7_p10.predict(X_ts_xgb)), 0, None)

m7_p99_raw = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.99, n_estimators=300, max_depth=6, learning_rate=0.03, random_state=42, n_jobs=-1)
m7_p99_raw.fit(X_cb_xgb, y_cb_raw)
pred_p99_raw = np.clip(m7_p99_raw.predict(X_ts_xgb), 0, None)

m7_p50_raw = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.50, n_estimators=300, max_depth=6, learning_rate=0.03, random_state=42, n_jobs=-1)
m7_p50_raw.fit(X_cb_xgb, y_cb_raw)
pred_p50_raw = np.clip(m7_p50_raw.predict(X_ts_xgb), 0, None)

r_std = X_ts_xgb['rolling_std_7'].values
surge_buffer = np.maximum(pred_p99_raw - pred_p50_raw, 2.0 * r_std + 2.0)
pred_p90 = pred_p50 + surge_buffer

m7_test_pred = pred_p50
test_metrics = evaluate_metrics(test_series, m7_test_pred)

print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 7: XGBOOST QUANTILE (OPTUNA FINE-TUNED P50) ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_XGB_Quantile': m7_test_pred, 'P10': pred_p10, 'P90': pred_p90}).to_csv('m7_xgb_quantile_preds.csv', index=False)


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 7: XGBOOST QUANTILE (OPTUNA FINE-TUNED P50) ===
  * RMSLE     : 0.5278
  * RMSE      : 4.3057
  * MAE       : 3.3294
  * WAPE (%)  : 38.8912
